1. Package Installation

In [2]:
# ============================================================
# 1. Package Installation
# ============================================================

!uv pip install -q transformers accelerate pillow matplotlib numpy
# !pip install -q transformers accelerate pillow matplotlib numpy

2. Import libraries

In [3]:
# ============================================================
# 2. Import Libraries
# ============================================================

import os
import re
import random
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# Add more packages if you want.

KeyboardInterrupt: 

3. Maze environment

In [ ]:
# ============================================================
# 3. Maze Environment
# ============================================================

class MazeEnv:
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (4, 4)

        # Obstacles can be changed by students

        """
        # 1. Easy
        self.obstacles = {
            (1, 1),
            (1, 2),
            (2, 3),
            (3, 1),
        }


        # 2. Medium
        self.obstacles = {
            (0, 2),
            (1, 2),
            (3, 0),
            (3, 1),
            (3, 3),
        }
        """

        # 3. Hard
        self.obstacles = {
            (0, 3),
            (1, 1),
            (1, 3),
            (2, 1),
            (3, 1),
            (3, 3),
        }


        self.actions = ["UP", "DOWN", "LEFT", "RIGHT"]
        self.reset()

    def reset(self):
        self.agent_pos = self.start
        return self.agent_pos

    def is_valid_position(self, pos):
        r, c = pos

        if r < 0 or r >= self.size:
            return False
        if c < 0 or c >= self.size:
            return False
        if pos in self.obstacles:
            return False

        return True

    def step(self, action):
        r, c = self.agent_pos

        if action == "UP":
            next_pos = (r - 1, c)
        elif action == "DOWN":
            next_pos = (r + 1, c)
        elif action == "LEFT":
            next_pos = (r, c - 1)
        elif action == "RIGHT":
            next_pos = (r, c + 1)
        else:
            next_pos = self.agent_pos

        if not self.is_valid_position(next_pos):
            next_pos = self.agent_pos
            reward = -5
            done = False
        elif next_pos == self.goal:
            self.agent_pos = next_pos
            reward = 10
            done = True
        else:
            self.agent_pos = next_pos
            reward=1
            done=False

    # FILL MORE CODE HERE TO MAKE IT WORK
    # CHANGE THE STRUCTURE OF THE CODE IF YOU WANT

        return next_pos, reward, done

4. Load VLM

In [ ]:
# ============================================================
# 4. Maze Visualization
# ============================================================

def draw_maze(env, trajectory=None, cell_size=80):
    size = env.size
    img_size = size * cell_size
    image = Image.new("RGB", (img_size, img_size), "white")
    draw = ImageDraw.Draw(image)

    for r in range(size):
        for c in range(size):
            x0 = c * cell_size
            y0 = r * cell_size
            x1 = x0 + cell_size
            y1 = y0 + cell_size

            pos = (r, c)

            if pos in env.obstacles:
                fill = "black"
            elif pos == env.start:
                fill = "lightgreen"
            elif pos == env.goal:
                fill = "gold"
            else:
                fill = "white"

            draw.rectangle([x0, y0, x1, y1], fill=fill, outline="gray")

            draw.text((x0 + 5, y0 + 5), f"{pos}", fill="blue")

    if trajectory is not None:
        for idx, pos in enumerate(trajectory):
            r, c = pos
            x = c * cell_size + cell_size // 2
            y = r * cell_size + cell_size // 2
            draw.ellipse([x-10, y-10, x+10, y+10], fill="red")
            draw.text((x+8, y-8), str(idx), fill="red")

    r, c = env.agent_pos
    x = c * cell_size + cell_size // 2
    y = r * cell_size + cell_size // 2
    draw.ellipse([x-18, y-18, x+18, y+18], fill="red")

    return image


env = MazeEnv(size=5)
img = draw_maze(env)
plt.imshow(img)
plt.axis("off")
plt.show()

5. Q-table initialization

In [ ]:
# ============================================================
# 5. Load VLM
# ============================================================

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
).to(DEVICE)
# CHANGE THE STRUCTURE OF THE CODE IF YOU WANT

model.eval()

In [ ]:
# ============================================================
# 5. Q-table Initialization
# ============================================================

env = MazeEnv(size=5)

actions = env.actions
action_to_idx = {a: i for i, a in enumerate(actions)}

# Q-table shape: row x col x action
Q = np.zeros((env.size, env.size, len(actions)), dtype=np.float32)


random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


def get_next_position(state, action):
    r, c = state

    if action == "LEFT":
        return r - 1, c
    elif action == "RIGHT":
        return r + 1, c
    elif action == "UP":
        return r, c - 1
    elif action == "DOWN":
        return r, c + 1

    return state


# 차후 VQA와 이동에 필요한 Util함수들

# 현재 위치에서 할 수 있는 선책지를 얻는 함수
def get_valid_actions(env, state=None):
    if state is None:
        state = env.agent_pos
    valid_actions = []
    for ac in env.valid_actions:
        nxt_pos = get_next_position(state, ac)
        if env.valid_position(nxt_pos):
            valid_actions.append(ac)
    return valid_actions

# Q-table에서 가장 profit이 큰 걸 찾는 함수
def select_best_valid_action(env, Q, state=None):
    if state is None:
        state = env.agent_pos
    valid_actions = get_valid_actions(env, state)

    if len(valid_actions) == 0:
        return None

    r, c = state

    q_values = []
    for action in valid_actions:
        action_idx = action_to_idx[action]
        q_values.append(Q[r, c, action_idx])

    max_q = max(q_values)

    best_actions = [
        action
        for action, q in zip(valid_actions, q_values)
        if q == max_q
    ]

    return random.choice(best_actions)


def select_action_with_output_guard(env, Q, suggested_action, state=None):
    if state is None:
        state = env.agent_pos

    valid_actions = get_valid_actions(env, state)

    if suggested_action is not None:
        suggested_action = str(suggested_action).strip().upper()

    # VLM output이 정상이고, 현재 state에서 실제 이동 가능한 경우
    if suggested_action in valid_actions:
        return suggested_action

    # VLM output이 이상하거나 벽/범위 밖 이동이면 Q-table fallback
    return select_best_valid_action(env, Q, state)



# VLM의 output에서 우리가 원하는 action만 parsing하는 함수
def parse_action(raw_output, actions):
    if raw_output is None:
        return None

    text = str(raw_output).strip().upper()

    # actions 예: ["UP", "DOWN", "LEFT", "RIGHT"]
    action_map = {action.upper(): action for action in actions}

    # VLM이 딱 "RIGHT"처럼 답한 경우
    if text in action_map:
        return action_map[text]

    # 문장 안에서 action 단어 찾기
    # 예: "I think the best action is RIGHT."
    tokens = re.findall(r"[A-Z]+", text)

    matched_actions = []

    for token in tokens:
        if token in action_map and token not in matched_actions:
            matched_actions.append(token)

    # action이 하나만 명확하게 나온 경우만 사용
    if len(matched_actions) == 1:
        return action_map[matched_actions[0]]

    # 없거나 여러 개면 모호하므로 fallback 하게 None 반환
    return None



6. Prompt Construction

In [ ]:
# ============================================================
# 6. Prompt Construction
# ============================================================

def get_relevant_q_values(Q, state, actions):
    r, c = state
    q_values = {a: float(Q[r, c, i]) for i, a in enumerate(actions)}
    return q_values


def build_prompt(env, Q, previous_state=None):
    
    # FILL MORE CODE HERE TO MAKE IT WORK
    # CHANGE THE STRUCTURE OF THE CODE IF YOU WANT
    
    return prompt

7. VLM action generation

In [ ]:
# ============================================================
# 7. VLM Action Generation
# ============================================================

@torch.no_grad()
def ask_vlm_for_action(env, Q, max_new_tokens=10, previous_state=None):
    
    # 1. Draw current agent position on the uploaded maze image
    image = draw_maze(env)
    # 2. Build text prompt
    prompt = build_prompt(env, Q, previous_state=previous_state)
    
    # 3. Construct multimodal message
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

   
    # FILL MORE CODE HERE TO MAKE IT WORK
    # CHANGE THE STRUCTURE OF THE CODE IF YOU WANT

    return action, raw_output



# FILL MORE CODE HERE TO MAKE IT WORK
# CHANGE THE STRUCTURE OF THE CODE IF YOU WANT

8. Q-learning update

In [ ]:
# ============================================================
# 8. Q-learning Update
# ============================================================

def update_q_table(Q, state, action, reward, next_state, alpha, gamma):
    
    # FILL MORE CODE HERE TO MAKE IT WORK
    # CHANGE THE STRUCTURE OF THE CODE IF YOU WANT

    return Q

9. Training Loop

In [ ]:
from tqdm.auto import tqdm
import time

# ============================================================
# 9. Training Loop
# ============================================================

num_episodes = 100 # Change this number if you need.
max_steps_per_episode = 50 # Change this number if you need.

episode_rewards = []
episode_steps = []
episode_success = []
episode_losses = []
vlm_q_alignment_logs = []


# FILL MORE CODE HERE TO MAKE IT WORK
# CHANGE THE STRUCTURE OF THE CODE IF YOU WANT


print("Training finished.")

10. Plot training results and Demo after training

In [ ]:
# ============================================================
# 10. Plot Training Results and Demo after training
# ============================================================

# FILL MORE CODE HERE TO MAKE IT WORK
# CHANGE THE STRUCTURE OF THE CODE IF YOU WANT